In [2]:
csv_folder = "C:/Program Files (x86)/Steam/steamapps/common/Darkest Dungeon® II/Darkest Dungeon II_Data/StreamingAssets/Excel/"

import csv, sys, random, subprocess
from pathlib import Path
from pympler import asizeof
import pandas as pd

random.seed(0)

class Element:
    def __init__(self, id, type, data_lines=[]):
        self.id = id
        self.type = type
        self.data = list(data_lines)

    def append(self, data_line):
        self.data.append(data_line)

    def get(self, field_name):
        return [line for line in self.data if line.name == field_name]

    def __repr__(self):
        repr = f"{self.id},{self.type}\n"
        for line in self.data:
            repr += f"{line}\n"
        if not len(self.data):
            repr += "\tempty\n"
        return repr
    
class ElementDataLine:
    def __init__(self, name, values):
        self.name = name
        self.values = []

        for v in values:
            if v == "True":
                self.values.append(True)
            elif v == "False":
                self.values.append(False)
            else:
                try:
                    self.values.append(float(v))
                except ValueError:
                    self.values.append(v)

    def __repr__(self):
        strvals = map(lambda v: str(v).rstrip('0').rstrip('.'), self.values)
        return f"\t{self.name},{','.join(strvals)},"

def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False
    
def number_if_is_number(s):
    try:
        return float(s)
    except ValueError:
        return None
    
def find_all_elements(csv_folder):
    def parse_file(filepath):
        with open(filepath.resolve(), "r", encoding="utf-8-sig") as f:
            return list(parse_rows(csv.reader(f)))

    def parse_rows(rows):
        current_element = None
        for row in remove_empty(rows):
            first_word = row[0]
            if first_word == "element_start":
                current_element = Element(row[1], row[2])
            elif first_word == "element_end":
                yield current_element
            else:
                values = remove_empty(row[1:])
                current_element.append(ElementDataLine(first_word, values))
                
    csv_filepaths = Path(csv_folder).rglob("*.csv")
    return [el for fp in csv_filepaths for el in parse_file(fp)]

def remove_empty(list):
    return filter(lambda x: x is not None and str(x).strip() != "", list)

def str_sizeof(x):
    return f"{asizeof.asizeof(x) / 1024 / 1024:.2f} MB"

elements = find_all_elements(csv_folder)
print(f"found {len(elements)} elements")
# print(str_sizeof(elements))
print(*random.sample(elements, k=3), "...", sep="")

del csv_folder

found 31219 elements
songbook_amorous,Unlock
	empty
spider_disease_10pct,Effect
	m_Chance,0.1,
	m_QuirkAddTag,disease_spider,
	m_QuirkAddAmount,1,
	m_QuirkAddAmountRange,0,
assist_ascetic_1,StoryChoice
	m_AnyTags,Assist,
	m_DrawTags,assist_destitute_basic,assist_militia_basic,assist_provisions_basic,
	m_Chance,10,
	m_CostId,story_relics_cost_med,
	m_AlignmentId,DONATE,
	m_PlayerStoryChoicePreviewIds,icon_story_relic_loot_Preview,icon_story_torch_Preview,
	m_PlayerStoryChoicePreviewValues,-12,45,
	m_PlayerStoryChoicePreviewShowNumbers,True,True,
	all_conditions,performer_has_ascetic,story_cost_condition_relics_med,
	m_ResultType,DRIVING,
	m_ResultLootIds,assist_reward_improved_torch,
...


In [ ]:
import ezodf

output_file = "./output.ods"

def filter_elements_by_type(element_type):
    return filter(lambda x: x.type == element_type, elements)

def find_all_fields_for_a_type(element_type):
    matching_elements = filter_elements_by_type(element_type)
    fields = set(f.name for el in matching_elements for f in el.data)
    return sorted(remove_empty(fields))

def find_all_lines_by_element_type_and_field_name(element_type, field_name) -> set:
    matching_elements = filter_elements_by_type(element_type)
    matching_lines = [line for el in matching_elements for line in el.get(field_name)]
    return matching_lines
    
def find_all_values_for_a_field(element_type, field_name) -> set:
    matching_lines = find_all_lines_by_element_type_and_field_name(element_type, field_name)
    return set(v for line in matching_lines for v in line.values)

def write_to_ods():
    ods = ezodf.newdoc(doctype="ods", filename=output_file)
    header = ["Field Name", "Input Type", "Marks", "Comment", "Values",]
    all_types = set(elem.type for elem in elements)

    for t in sorted(all_types):
        fields = find_all_fields_for_a_type(t)
        sheet = ezodf.Sheet(t, size=(len(fields) + 1, len(header)))
        ods.sheets.append(sheet)
        for i, h in enumerate(header):
            sheet[0, i].set_value(h)
        for i, f in enumerate(fields):
            values = find_all_values_for_a_field(t, f)
            values = [f"{item:g}" if isinstance(item, (int, float)) and not isinstance(item, bool) else str(item) for item in values]
            sheet[i + 1, 0].set_value(f)
            sheet[i + 1, 4].set_value(','.join(values))

    ods.save()

write_to_ods()
print(f"Wrote data to {output_file}")

Wrote data to ./output.ods


In [7]:
odf_filepath = "..\CSV Description.ods"

def odf_to_md(first_n=None, file=None):
    all_sheets = pd.read_excel(odf_filepath, engine="odf", sheet_name=None)
    sheet_names = sorted(all_sheets.keys())
    header = ["Field Name", "Input Type", "Marks", "Comment",]
    for i, sheet_name in enumerate(sheet_names):
        if first_n and (i >= first_n):
            break
        df = all_sheets[sheet_name]
        print(f"## Sheet: {sheet_name}\n\n", file=file)
        df_filtered = df[[col for col in header if col in df.columns]].fillna("")
        print(df_filtered.to_markdown(index=False, tablefmt="github"), file=file)
        print("\n\n---\n\n", file=file)

odf_to_md(2)

## Sheet: Achievement


| Input Type   | Marks    | Comment                                                            |
|--------------|----------|--------------------------------------------------------------------|
| tag[]        | receiver |                                                                    |
| tag[]        | receiver |                                                                    |
| tag[]        | receiver |                                                                    |
| tag[]        | receiver | m_sourceTags and m_destinationTags must both be empty or not empty |
| tag[]        | receiver |                                                                    |
| int          |          | Incompatible with m_targetInt and m_targetFloat                    |
| tag[]        | receiver |                                                                    |
| float        |          | Can't have both m_targetInt and m_targetFloat defined              |
| int 